In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

In [13]:
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

response = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="nova",
    input="We are launching a new product today.",
    instructions="Speak slowly, like a professional announcer. Be confident and clear."
)

response.stream_to_file("announcement.mp3")

/tmp/ipykernel_467418/3407431198.py:12: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file("announcement.mp3")


In [15]:
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

response = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input="Tired of waiting for opportunities? Discover the new SmartX earbuds — immersive sound, all-day battery, and premium comfort.",
    instructions="Speak like a high-energy ad voice. Confident, exciting, slightly fast-paced."
)

response.stream_to_file("ad.mp3")

/tmp/ipykernel_467418/2023192019.py:12: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file("ad.mp3")


In [18]:
from openai import OpenAI
import os
import re

openai_api_key = os.environ.get("OPENAI_API_KEY", "your-api-key-here")
client = OpenAI(api_key=openai_api_key)

TTS_CHAR_LIMIT = 4096


def split_text(text: str, max_chars: int = TTS_CHAR_LIMIT) -> list[str]:
    """
    Split text into chunks at sentence boundaries, keeping each chunk
    under max_chars. Falls back to splitting at word boundaries if a
    single sentence is too long.
    """
    # Split on sentence-ending punctuation followed by whitespace
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    current = ""

    for sentence in sentences:
        # If a single sentence exceeds the limit, break it at word boundaries
        if len(sentence) > max_chars:
            words = sentence.split()
            for word in words:
                if len(current) + len(word) + 1 > max_chars:
                    if current:
                        chunks.append(current.strip())
                    current = word + " "
                else:
                    current += word + " "
        elif len(current) + len(sentence) + 1 > max_chars:
            chunks.append(current.strip())
            current = sentence + " "
        else:
            current += sentence + " "

    if current.strip():
        chunks.append(current.strip())

    return chunks


def text_to_speech(
    text: str,
    output_file: str = "output.mp3",
    model: str = "gpt-4o-mini-tts",
    voice: str = "coral",
    instructions: str = "Speak like a high-energy ad voice. Confident, exciting, slightly fast-paced.",
) -> None:
    """
    Convert arbitrarily long text to speech, chunking as needed,
    and write the result to output_file.
    """
    chunks = split_text(text)
    print(f"Text split into {len(chunks)} chunk(s).")

    audio_segments: list[bytes] = []

    for i, chunk in enumerate(chunks, 1):
        print(f"  Generating chunk {i}/{len(chunks)} ({len(chunk)} chars)…")
        response = client.audio.speech.create(
            model=model,
            voice=voice,
            input=chunk,
            instructions=instructions,
        )
        audio_segments.append(response.content)

    # Concatenate raw MP3 bytes and write once
    with open(output_file, "wb") as f:
        for segment in audio_segments:
            f.write(segment)

    print(f"Saved to {output_file}")


# ── Main ──────────────────────────────────────────────────────────────────────

long_text = """
HOOK 3 (Third Option):
Most online store programs work like this: they get paid when you buy their course. Mine is different. I only get paid when *your* store actually moves inventory.
---
BODY:
PAIN:
Look, I understand if you're skeptical. You've been right to be before. Maybe you've tried one of these online store things and it just felt like a casino. You saw the red flags, hoped it would be different, and now feel a little stupid for trusting your judgment. It cost you a few thousand dollars and maybe some pride. Your spouse might even be wondering if you'll follow through. You're asking, "What's the catch *this* time?"
PROMISE:
The truth is, it wasn't your judgment that was wrong. It was the model they were selling you. What if there's a different way to build a real online storefront, selling products already in demand? A way to earn supplemental income without the upfront gamble or the stress of finding products. A way to build quiet confidence, and a little more breathing room. An earned calm.
PROOF:
I'm Marcus Vane. I ran wholesale logistics for 14 years before starting the Sovereign Storefront program 5 years ago. I saw warehouses packed with perfectly good returned inventory, and regular folks trying to sell online without knowing what to get, or where.
My system helps you build an online consignment storefront, reselling that inventory from our vetted regional liquidator partners. We provide a weekly 'Liquidator Match Sheet' – inventory with existing demand, ready to be cataloged. The difference? I only get paid when *your* store moves their products. Liquidators pay *me* a placement fee, so my income is tied directly to your success. It's a completely different incentive.
As member Dana R. said, "I was sure this was another version of the thing that cost me $2,400. Then he explained who actually pays him and it clicked. I have a real store now, not a folder of unwatched videos." With over 9,400 members globally, and these match sheets running for five years, this is a real operational system, not a get-rich-quick scheme.
CTA:
If you're ready to see how this operational system actually works, without the hype or empty promises, watch our First Pallet workshop. It's a two-hour deep dive into the mechanism, including a sample match sheet and store starter kit. It costs just $1. The link is below.
""".strip()

text_to_speech(long_text, output_file="ad.mp3")

Text split into 1 chunk(s).
  Generating chunk 1/1 (2348 chars)…
Saved to ad.mp3
